# Task 3 - Monthly Spending Prediction (Regression, metric: NMAE)

CPE342 Karena Player Intelligence System

> **Rules followed:** models are trained from scratch (no pre-trained models, no AutoML); every statistic (blend weight, hurdle cut-off, number of trees) is chosen on **train** out-of-fold predictions only; the test set is used only for the final prediction.

The target is heavily skewed with many zeros, and MAE is minimised by the conditional **median**, so the models use an **L1 objective**. Two models are blended: a plain L1 LightGBM and a *hurdle* model (classifier for P(pay) + L1 regressor trained on payers only).

## 1. Import Libraries & Configuration

In [ ]:
import os, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingClassifier
warnings.filterwarnings("ignore")

SEED = 42
QUICK = os.environ.get("QUICK") == "1"   # QUICK=1 -> small smoke test (results not meaningful)
ROOT = Path.cwd()                         # repo root, or the task3/ folder itself
OUT = ROOT / ("outputs_quick" if QUICK else "outputs")
(OUT / "models").mkdir(parents=True, exist_ok=True)
print("QUICK mode:", QUICK)

TASK, ID, TARGET = 3, "player_id", "spending_30d"
N_FOLDS = 3 if QUICK else 5
DATA = ROOT / "task3" if (ROOT / "task3" / "train.csv").exists() else ROOT   # run from repo root or from inside task3/

In [ ]:
# ---------- helper functions ----------
def nmae(y, p):
    # ASSUMPTION: NMAE = MAE / mean(y). Edit here if the task defines it differently
    # (e.g. divide by median(y) or by range).
    return np.mean(np.abs(y - p)) / np.mean(y)

def add_features(df):
    d = df.copy()
    d["spend_ratio"] = d["prev_month_spending"] / (d["historical_spending"] + 1)
    d["spend_per_day"] = d["historical_spending"] / (d["account_age_days"] + 1)
    d["txn_per_day"] = d["total_transactions"] / (d["account_age_days"] + 1)
    d["recency_x_freq"] = d["purchase_frequency"] / (d["days_since_last_purchase"] + 1)
    d["playtime_per_day"] = d["total_playtime_hours"] / (d["account_age_days"] + 1)
    d["no_history"] = ((d["historical_spending"].fillna(0) == 0) & (d["prev_month_spending"].fillna(0) == 0)).astype(int)
    d["log_prev"] = np.log1p(d["prev_month_spending"].clip(lower=0))
    d["log_hist"] = np.log1p(d["historical_spending"].clip(lower=0))
    return d.replace([np.inf, -np.inf], np.nan)

def make_l1(n_estimators=2000):
    return lgb.LGBMRegressor(objective="l1", n_estimators=n_estimators, learning_rate=0.03, num_leaves=63,
                             min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                             random_state=SEED, verbose=-1)

def fit_l1(Xtr, ytr, Xva=None, yva=None, n_estimators=2000):
    m = make_l1(n_estimators)
    if Xva is not None:
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="l1", callbacks=[lgb.early_stopping(100, verbose=False)])
    else:
        m.fit(Xtr, ytr)
    return m

def fit_hurdle(Xtr, ytr):
    """Classifier P(pay) + L1 regressor trained on payers only."""
    pay = ytr > 1.0
    clf = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=SEED).fit(Xtr, pay.astype(int))
    reg = fit_l1(Xtr[pay], ytr[pay], n_estimators=500)
    def predict(X, cutoff=0.5):
        p = clf.predict_proba(X)[:, 1]
        return np.where(p >= cutoff, np.clip(reg.predict(X), 0, None), 0.0)
    return predict, clf, reg

def save_outputs(task, pred, model_obj, extra=None):
    df = pd.DataFrame({"id": [f"ANS{i + 1:05d}" for i in range(len(pred))], "pred": pred})
    for k, v in (extra or {}).items():
        df[k] = v
    df.to_csv(OUT / f"task{task}_pred.csv", index=False)
    joblib.dump(model_obj, OUT / "models" / f"task{task}_model.joblib")
    print(f"saved {len(df)} predictions -> {OUT / f'task{task}_pred.csv'}")
    print(f"saved model -> {OUT / 'models' / f'task{task}_model.joblib'}")

## 2. Load Data

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
if QUICK:
    train = train.sample(20000, random_state=SEED).reset_index(drop=True)
print(f"Train shape: {train.shape}\nTest shape:  {test.shape}")
train.head()

## 3. Exploratory Data Analysis (EDA)

EDA uses the **train** set only.

In [ ]:
y_all = train[TARGET]
print(y_all.describe().round(2).to_string())
print(f"share of zeros: {(y_all == 0).mean():.1%}")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(y_all, bins=60, color="steelblue"); ax[0].set_title(TARGET)
ax[1].hist(np.log1p(y_all), bins=60, color="tomato"); ax[1].set_title(f"log1p({TARGET})")
plt.show()

In [ ]:
miss = (train.drop(columns=[TARGET]).isna().mean() * 100).sort_values(ascending=False)
print("max missing %:", round(miss.max(), 1))
miss[miss > 0].head(25).sort_values().plot(kind="barh", color="tomato", figsize=(7, 6), title="Missing values % (train, top 25)")
plt.show()

In [ ]:
feats = train.drop(columns=[ID, TARGET]).select_dtypes("number")
top = feats.corrwith(train[TARGET], method="spearman").abs().fillna(0).sort_values(ascending=False).head(15)
print("most relevant features:", list(top.index[:10]))
top.sort_values().plot(kind="barh", figsize=(7, 5), color="steelblue", title="|Spearman| with target")
plt.show()

In [ ]:
cols = list(top.index[:6])
s = train.sample(min(5000, len(train)), random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.flat, cols):
    ax.scatter(s[c], s[TARGET], s=2, alpha=0.3); ax.set_title(c); ax.set_ylabel(TARGET)
plt.tight_layout(); plt.show()

In [ ]:
hm = feats[list(top.index[:12])].copy(); hm[TARGET] = train[TARGET]
c = hm.corr()
plt.figure(figsize=(9, 7)); plt.imshow(c, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(c)), c.columns, rotation=60, ha="right"); plt.yticks(range(len(c)), c.columns)
plt.title("Correlation (top features + target)"); plt.tight_layout(); plt.show()

## 4. Data Preprocessing

- the target is **not** clipped or transformed (MAE on the raw target is what is scored)
- engineered ratios: spending per day / transaction rate / playtime per day, previous month vs history, recency vs frequency, `no_history` flag, `log1p` of both spending columns
- missing values are left as NaN (LightGBM / HistGradientBoosting handle them natively); `inf` from ratios is turned into NaN

In [ ]:
X_all = add_features(train.drop(columns=[ID, TARGET]))
y = train[TARGET].values
X_test = add_features(test.drop(columns=[ID]))[X_all.columns]
print(f"features: {X_all.shape[1]}")
print("baselines (NMAE):")
print("  median     :", round(nmae(y, np.full(len(y), np.median(y))), 4))
print("  prev_month :", round(nmae(y, train["prev_month_spending"].fillna(0).values), 4))

## 5. Model Training & Validation

### 5.1 5-fold CV: L1 LightGBM vs hurdle model (metric: NMAE, lower is better)
The hurdle cut-off and the blend weight are chosen on the out-of-fold predictions.

In [ ]:
kf = KFold(N_FOLDS, shuffle=True, random_state=SEED)
cutoffs = np.round(np.linspace(0.3, 0.7, 9), 2)
oof_l1 = np.zeros(len(y))
oof_hur = {c: np.zeros(len(y)) for c in cutoffs}
best_iters = []

for k, (tr, va) in enumerate(kf.split(X_all)):
    Xtr, Xva, ytr, yva = X_all.iloc[tr], X_all.iloc[va], y[tr], y[va]
    m = fit_l1(Xtr, ytr, Xva, yva, n_estimators=200 if QUICK else 2000)
    best_iters.append(m.best_iteration_ or m.n_estimators)
    oof_l1[va] = np.clip(m.predict(Xva), 0, None)
    hur = fit_hurdle(Xtr, ytr)[0]
    for c in cutoffs:
        oof_hur[c][va] = hur(Xva, c)
    print(f"fold {k + 1}/{N_FOLDS}: L1 NMAE = {nmae(yva, oof_l1[va]):.4f}")

best_c = min(cutoffs, key=lambda c: nmae(y, oof_hur[c]))
best_w, best_s = 1.0, nmae(y, oof_l1)
for w in np.linspace(0, 1, 11):
    s = nmae(y, w * oof_l1 + (1 - w) * oof_hur[best_c])
    if s < best_s:
        best_w, best_s = w, s
oof = np.clip(best_w * oof_l1 + (1 - best_w) * oof_hur[best_c], 0, 300000)

print("\n=== CV (out-of-fold) ===")
print("L1 model :", round(nmae(y, oof_l1), 4))
print(f"Hurdle   : {nmae(y, oof_hur[best_c]):.4f} (cutoff={best_c})")
print(f"Blend    : {best_s:.4f} (weight L1={best_w:.1f})")

### 5.2 Error analysis (out-of-fold)

In [ ]:
grp = pd.cut(y, [-1, 0, 1000, 20000, 1e9], labels=["zero", "low<=1k", "mid<=20k", "whale>20k"])
bd = pd.DataFrame({"grp": grp, "ae": np.abs(y - oof)}).groupby("grp", observed=True)["ae"]
print("Share of total abs error by group:")
print((bd.sum() / bd.sum().sum()).round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
s = np.random.RandomState(SEED).choice(len(y), min(8000, len(y)), replace=False)
ax[0].scatter(np.log1p(y[s]), np.log1p(oof[s]), s=3, alpha=0.3)
lim = np.log1p(y.max()); ax[0].plot([0, lim], [0, lim], "r--")
ax[0].set_xlabel("log1p(actual)"); ax[0].set_ylabel("log1p(OOF prediction)"); ax[0].set_title("Actual vs predicted (out-of-fold)")
ax[1].hist(np.clip(oof - y, -20000, 20000), bins=80, color="steelblue"); ax[1].set_title("Residuals (pred - actual, clipped to +-20k)")
plt.show()

## 6. Predict on the Test Set & Save

In [ ]:
n_final = int(np.mean(best_iters) * 1.1)
final_l1 = fit_l1(X_all, y, n_estimators=n_final)
hur_final, hur_clf, hur_reg = fit_hurdle(X_all, y)

p_l1 = np.clip(final_l1.predict(X_test), 0, None)
p_hur = hur_final(X_test, best_c)
pred = np.clip(best_w * p_l1 + (1 - best_w) * p_hur, 0, 300000)

print(f"test prediction: min {pred.min():.1f} | mean {pred.mean():.1f} | max {pred.max():.1f}   (train target mean {y.mean():.1f}, max {y.max():.0f})")
plt.hist(np.log1p(y), bins=60, alpha=0.5, density=True, label="train target")
plt.hist(np.log1p(pred), bins=60, alpha=0.5, density=True, label="test prediction")
plt.xlabel("log1p(spending)"); plt.legend(); plt.title("Target vs predicted distribution"); plt.show()
save_outputs(TASK, pred, {"l1_model": final_l1, "hurdle_clf": hur_clf, "hurdle_reg": hur_reg, "cutoff": best_c,
                          "blend_weight_l1": best_w, "features": list(X_all.columns)})